# DSE ZG532 - Assignment 1
## Retail Sales Performance Analysis

**Student ID:** 2026nd03247  
**Name:** SAUMYA

Task 1 - Dataset Selection and Ingestion

## Dataset metadata

| Field | Details |
|-------|---------|
| Dataset name | Sample Superstore Sales |
| Stable public source URL | https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv |
| Data provider | Tableau Sample Data (mirrored on GitHub) |
| Access date | 05-Sep-2026 |
| Domain | Retail / Sales |
| File format | CSV |
| Initial records | 10800 |
| Initial attributes | 21 |

### Important column descriptions

| Column | Description |
|--------|-------------|
| Order Date, Ship Date | Date the order was placed and shipped |
| Ship Mode | Delivery method (Standard Class, Second Class, etc.) |
| Segment | Customer type - Consumer, Corporate, Home Office |
| Region, State, City | Geographic location of the order |
| Category, Sub-Category | Product grouping |
| Product Name | Name of item sold |
| Sales | Revenue from the line item |
| Quantity | Units sold |
| Discount | Discount applied (0 to 1) |
| Profit | Profit earned on the line item |

### Practical data questions

1. Which product categories generate high sales but low profit margins?
2. How does average profit differ across regions and customer segments?
3. Do higher discounts reduce profit, and does that change by ship mode?

In [1]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv"
FILE_TYPE = "csv"

In [2]:
dataset_meta = {
    "name": "Sample Superstore Sales",
    "url": DATA_URL,
    "provider": "Tableau Sample Data",
    "access_date": "2026-09-05",
    "domain": "Retail / Sales",
    "format": FILE_TYPE,
}

col_desc = {
    "Order Date": "date order was placed",
    "Ship Date": "date order was shipped",
    "Ship Mode": "delivery type",
    "Segment": "customer segment",
    "Region": "US sales region",
    "Category": "product category",
    "Sub-Category": "product sub-category",
    "Sales": "revenue amount",
    "Quantity": "units sold",
    "Discount": "discount fraction",
    "Profit": "profit amount",
}

key_cols = ["Order Date", "Sales", "Profit", "Category", "Region", "Segment", "Discount", "Quantity"]
cat_cols = ("Category", "Sub-Category", "Segment", "Region")
num_cols = {"Sales", "Profit", "Quantity", "Discount"}
regions = {"East", "West", "Central", "South"}

questions = [
    "Which categories have high sales but low profit margin?",
    "How does profit vary by region and segment?",
    "Does discount level affect profit across ship modes?",
]

print("DATASET METADATA")
print("-" * 40)
for k, v in dataset_meta.items():
    print(f"{k:15s}: {v}")

print("\nImportant columns:")
for col, desc in col_desc.items():
    print(f"  {col:15s} - {desc}")

print("\nPractical data questions:")
for i, q in enumerate(questions, 1):
    print(f"  {i}. {q}")

DATASET METADATA
----------------------------------------
name           : Sample Superstore Sales
url            : https://raw.githubusercontent.com/leonism/sample-superstore/master/data/superstore.csv
provider       : Tableau Sample Data
access_date    : 2026-09-05
domain         : Retail / Sales
format         : csv

Important columns:
  Order Date      - date order was placed
  Ship Date       - date order was shipped
  Ship Mode       - delivery type
  Segment         - customer segment
  Region          - US sales region
  Category        - product category
  Sub-Category    - product sub-category
  Sales           - revenue amount
  Quantity        - units sold
  Discount        - discount fraction
  Profit          - profit amount

Practical data questions:
  1. Which categories have high sales but low profit margin?
  2. How does profit vary by region and segment?
  3. Does discount level affect profit across ship modes?


In [3]:
def load_and_check(url, expected_cols=None):
    """Load csv from url and do basic validation. Returns dataframe and a status dict."""
    df = pd.read_csv(url)
    
    status = {
        "rows": len(df),
        "cols": len(df.columns),
        "missing_cols": [],
        "ok": True,
    }
    
    if expected_cols is not None:
        for col in expected_cols:
            if col not in df.columns:
                status["missing_cols"].append(col)
                status["ok"] = False
    
    return df, status

In [4]:
print(f"Loading {FILE_TYPE.upper()} from URL using pd.read_csv...")
raw_df, load_status = load_and_check(DATA_URL, expected_cols=key_cols)

if load_status["ok"]:
    print("All expected columns found")
else:
    print("Missing:", load_status["missing_cols"])

print(f"Loaded {load_status['rows']} rows, {load_status['cols']} columns")

Loading CSV from URL using pd.read_csv...


All expected columns found
Loaded 10800 rows, 21 columns


In [5]:
issues = []

for col in key_cols:
    if col not in raw_df.columns:
        issues.append(f"missing column: {col}")

row_count = len(raw_df)
col_count = len(raw_df.columns)

if row_count < 1000:
    issues.append("dataset seems too small")
elif row_count > 50000:
    issues.append("dataset larger than expected")

for col in num_cols:
    if col in raw_df.columns and raw_df[col].dtype not in ["float64", "int64"]:
        issues.append(f"{col} not numeric (dtype={raw_df[col].dtype})")

if "Region" in raw_df.columns:
    found_regions = set(raw_df["Region"].dropna().unique())
    unknown = found_regions - regions
    if len(unknown) > 0:
        issues.append(f"unexpected regions: {unknown}")

null_sales = raw_df["Sales"].isnull().sum() if "Sales" in raw_df.columns else 0
if null_sales > 0:
    issues.append(f"{null_sales} rows with missing Sales")

print("VALIDATION RESULTS")
print("-" * 30)
print(f"Rows: {row_count}  |  Columns: {col_count}")
print(f"Key columns present: {load_status['ok']}")

if len(issues) == 0:
    print("No issues found")
else:
    print(f"Issues found ({len(issues)}):")
    for iss in issues:
        print(f"  - {iss}")

VALIDATION RESULTS
------------------------------
Rows: 10800  |  Columns: 21
Key columns present: True
Issues found (1):
  - 806 rows with missing Sales


### Dataset structure

In [6]:
print("Shape (rows, columns):", raw_df.shape)
print(f"Rows: {raw_df.shape[0]}, Attributes: {raw_df.shape[1]}")
print("\nColumn names:")
for i, c in enumerate(raw_df.columns, 1):
    print(f"  {i:2d}. {c}")

Shape (rows, columns): (10800, 21)
Rows: 10800, Attributes: 21

Column names:
   1. Row ID
   2. Order ID
   3. Order Date
   4. Ship Date
   5. Ship Mode
   6. Customer ID
   7. Customer Name
   8. Segment
   9. Country
  10. City
  11. State
  12. Postal Code
  13. Region
  14. Product ID
  15. Category
  16. Sub-Category
  17. Product Name
  18. Sales
  19. Quantity
  20. Discount
  21. Profit


### Dataset preview

In [7]:
raw_df.head(10)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164
5,6,CA-2015-115812,6/9/2015,6/14/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7.0,0.00,14.1694
6,7,CA-2015-115812,6/9/2015,6/14/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800,4.0,0.00,1.9656
7,8,CA-2015-115812,6/9/2015,6/14/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6.0,0.20,90.7152
8,9,CA-2015-115812,6/9/2015,6/14/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3.0,0.20,5.7825
9,10,CA-2015-115812,6/9/2015,6/14/2015,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032.0,West,OFF-AP-10002892,Office Supplies,Appliances,Belkin F5C206VTEL 6 Outlet Surge,114.9000,5.0,0.00,34.4700


### Data types

In [8]:
raw_df.dtypes

Row ID               str
Order ID             str
Order Date           str
Ship Date            str
Ship Mode            str
Customer ID          str
Customer Name        str
Segment              str
Country              str
City                 str
State                str
Postal Code      float64
Region               str
Product ID           str
Category             str
Sub-Category         str
Product Name         str
Sales            float64
Quantity         float64
Discount         float64
Profit           float64
dtype: object

In [9]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10800 entries, 0 to 10799
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         10800 non-null  str    
 1   Order ID       10800 non-null  str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9983 non-null   float64
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       99

### Memory usage

In [10]:
mem = raw_df.memory_usage(deep=True)
print(mem)
print(f"\nTotal: {mem.sum() / 1024**2:.2f} MB")

Index               132
Row ID           570538
Order ID         680350
Order Date       606104
Ship Date        606206
Ship Mode        643652
Customer ID      595450
Customer Name    646451
Segment          603819
Country          645420
City             608749
State            600319
Postal Code       86400
Region           564063
Product ID       665408
Category         643447
Sub-Category     587372
Product Name     891472
Sales             86400
Quantity          86400
Discount          86400
Profit            86400
dtype: int64

Total: 10.10 MB


### Suitability inference

The dataset is suitable for further analysis. It loaded 10800 rows with 21 columns from the public URL without any login. All key columns needed for sales and profitability analysis are present - Sales, Profit, Discount, Category, Region and Segment. The data has a mix of categorical fields (Segment, Region, Category) and numeric fields (Sales, Profit, Quantity) which is good for aggregation and numerical work later. Order Date and Ship Date came in as strings so they will need conversion, and some rows have missing Sales values that should be checked. Overall the structure matches what we need for a retail sales analysis.